# 8 · Unsteady problems — the double-glazing flow

Real processes **evolve in time**. The recipe is almost always the same: 
* discretise **space** as before (a weak form, an FE space), then march **time** in small steps.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from ngsolve import *
from netgen.occ import WorkPlane, OCCGeometry, X, Y
from ngsolve.webgui import Draw

## 1. The double-glazing problem

On the square $\Omega=(-1,1)^2$ a temperature $u(t,\mathbf x)$ is **heated** on the right wall
($u=1$), held **cold** on the left ($u=0$), and the top & bottom are **insulated**. A fixed
**recirculating wind** $\mathbf b$ stirs it:
$$ \partial_t u \;+\; \mathbf b\!\cdot\!\nabla u \;-\; \varepsilon\,\Delta u \;=\; 0,
   \qquad \mathbf b(x,y)=\bigl(2y(1-x^2),\,-2x(1-y^2)\bigr). $$
The wind runs **clockwise** in a single cell; with small diffusion $\varepsilon$ the heat is
**carried around** before it spreads.

![The square (-1,1)² — hot right wall u=1, cold left wall u=0, insulated top & bottom, and a clockwise recirculating wind](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/doubleglazing-domain.png)

In [ ]:
square = WorkPlane().MoveTo(-1, -1).Rectangle(2, 2).Face()              # Ω = (-1, 1)²
square.edges.Min(X).name = "left";   square.edges.Max(X).name = "right"
square.edges.Min(Y).name = "bottom"; square.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(square, dim=2).GenerateMesh(maxh=0.1))         # coarse — paired with high order below
wind = CF((2*y*(1-x*x), -2*x*(1-y*y)))                 # the recirculating draught
Draw(wind, mesh, "wind", vectors={"grid_size": 24})

## 2. Space, then time — implicit Euler

**Space** is the weak form on an `H1` space (Dirichlet on the hot/cold walls only), here on a
**coarse mesh at high order $k=4$** — far fewer elements than a fine low-order mesh for the
same accuracy (cheaper to march), and the high-order field draws *smoother*. We
assemble two operators: the **mass** matrix $M$ (from $\int u\,v$) and the **stiffness**
$A$ (diffusion **+** convection, $\int \varepsilon\nabla u\!\cdot\!\nabla v + (\mathbf b\!\cdot\!
\nabla u)\,v$). **Time** is then *implicit Euler*: from $\partial_t u + Au = 0$,
$$ \frac{u^{n+1}-u^n}{\Delta t} + A\,u^{n+1} = 0 \;\Longrightarrow\;
   \underbrace{(M+\Delta t\,A)}_{M^{*}}\,u^{n+1} = M\,u^n . $$
$M^{*}$ is **constant**, so we **factorise it once** and only back-substitute each step.

In [ ]:
eps, dt = 0.05, 0.01
fes = H1(mesh, order=4, dirichlet="right|left")        # high order k=4 (only hot/cold walls Dirichlet)
u, v = fes.TnT()
M = BilinearForm(u*v*dx).Assemble()
A = BilinearForm(eps*grad(u)*grad(v)*dx + (wind*grad(u))*v*dx).Assemble()
mstar = M.mat.CreateMatrix()
mstar.AsVector().data = M.mat.AsVector() + dt*A.mat.AsVector()   # M* = M + dt·A, assembled once
inv = mstar.Inverse(fes.FreeDofs())                              # factor once, reuse every step

gfu = GridFunction(fes)
def SetInitialValues(gf):
    gf.Set(CF(1), definedon=mesh.Boundaries("right"))     # hot right wall = 1, everything else 0
SetInitialValues(gfu)
Draw(gfu, mesh, "temperature", min=0, max=1, autoscale=False)

## 3. Step in time — watch the heat go around

One step is a single back-substitution: form the right-hand side $M u^n$ (equivalently the
residual update below, which keeps the Dirichlet walls fixed), solve, repeat. We collect a few
frames to animate the heat being **dragged down the hot wall and swept around** the cell.

In [ ]:
tend, res = 2.5, gfu.vec.CreateVector()
nsteps = int(tend / dt + 0.5)
anim = GridFunction(fes, multidim=0)                   # collect snapshots on the high-order mesh itself
anim.AddMultiDimComponent(gfu.vec)                     # t = 0
with TaskManager():
    for step in range(1, nsteps + 1):
        res.data = -dt * A.mat * gfu.vec               # implicit-Euler residual (Dirichlet kept)
        gfu.vec.data += inv * res
        if step % 30 == 0:                             # ~9 snapshots for the animation
            anim.AddMultiDimComponent(gfu.vec)
print(f"stepped to t={tend}:  temperature in [{min(gfu.vec):.2f}, {max(gfu.vec):.2f}]")

In [ ]:
Draw(anim, mesh, "temperature", min=0, max=1, autoscale=False,
     interpolate_multidim=True, animate=True, order=2)

## 4. Towards the steady state

Marched long enough, the transient settles to a **steady** recirculating temperature — the same
answer the stationary problem $\mathbf b\!\cdot\!\nabla u-\varepsilon\Delta u=0$ would give
directly. Implicit Euler is **unconditionally stable**, so we may take large steps to *reach*
that state cheaply; for *accuracy in time* one uses smaller steps or a higher-order scheme
(e.g. **Crank–Nicolson**, $\tfrac12$-implicit).

In [ ]:
Draw(gfu, mesh, "steady temperature", min=0, max=1, autoscale=False)

## Supplementary — the dt-syntax stepper, and a generic DIRK

We hand-rolled $M^{*}=M+\Delta t\,A$ above. NGSolve's **`ngsolve.timestepping`** library does
that bookkeeping for you: write the time derivative **straight into the weak form** as
**`u.dt`**, and hand the whole equation to a ready-made stepper — `ImplicitEuler`,
`CrankNicolson`, `Newmark`. Same implicit-Euler scheme as §2–3, no hand-rolled $M^{*}$; the
convection only makes the system non-symmetric, so we precondition with a `Direct` factorisation.

In [ ]:
from ngsolve import preconditioners
from ngsolve.timestepping import ImplicitEuler

equation = (u.dt * v + eps * grad(u) * grad(v) + (wind * grad(u)) * v) * dx   # ∂ₜu  ↦  u.dt

gfu2 = GridFunction(fes)
SetInitialValues(gfu2)

stepper = ImplicitEuler(equation, dt=dt, pc_cls=preconditioners.Direct)

with TaskManager():
    stepper.Integrate(gfu2, end_time=0.1)            # a few library steps — same march as §2–3
Draw(gfu, mesh, "steady temperature", min=0, max=1, autoscale=False)    

Under the hood a stepper simply **replaces** `u.dt` by $(u-u^n)/\Delta t$. We can **exploit the
same machinery** for a **generic diagonally-implicit Runge–Kutta (DIRK)** marcher: each stage is
an implicit-Euler-like solve with effective step $a_{ii}\Delta t$ and a known, accumulated
right-hand side. Driven by a **Butcher tableau** $(A,b)$ it works for *any* such `u.dt` equation:

In [ ]:
class DIRK:
    """A generic diagonally-implicit Runge–Kutta marcher for  ∫ u.dt·v + a(u,v) = 0, built
    straight from a dt-syntax `equation` + a Butcher tableau (A lower-triangular, b). Each stage
    reuses the dt-replacement:  u.dt ↦ (u − knownᵢ)/(a_ii·Δt)."""
    def __init__(self, equation, A, b, dt, fes):
        udt = next(p for p in equation.GetProxies() if p.dt_order == 1)
        self.A, self.b, self.s, self.dt, self.fes = A, b, len(b), dt, fes
        self.known = GridFunction(fes)                       # uⁿ + Δt Σ_{j<i} a_ij kⱼ
        self.stage = []                                      # one factorised stage system each
        for i in range(self.s):
            repl = {udt: 1 / (A[i][i] * dt) * (udt.anti_dt - udt.anti_dt.ReplaceFunction(self.known))}
            bf = BilinearForm(equation.Replace(repl)); bf.Assemble()
            self.stage.append((bf, bf.mat.Inverse(fes.FreeDofs())))
        self.k = [GridFunction(fes) for _ in range(self.s)]  # stage slopes kᵢ
    def Step(self, w):
        un = w.vec.CreateVector(); un.data = w.vec
        r = w.vec.CreateVector()
        for i in range(self.s):
            self.known.vec.data = un                         # accumulate the known right-hand side
            for j in range(i):
                self.known.vec.data += self.dt * self.A[i][j] * self.k[j].vec
            bf, inv = self.stage[i]
            w.vec.data = self.known.vec                      # Dirichlet-correct initial guess
            bf.Apply(w.vec, r); w.vec.data -= inv * r        # one linear solve → the stage value
            self.k[i].vec.data = 1 / (self.A[i][i] * self.dt) * (w.vec - self.known.vec)
        w.vec.data = un
        for i in range(self.s):                              # uⁿ⁺¹ = uⁿ + Δt Σ bᵢ kᵢ
            w.vec.data += self.dt * self.b[i] * self.k[i].vec

**Play it out** on the double-glazing with the 2-stage, L-stable **SDIRK2** tableau
($\gamma = 1-\tfrac{1}{\sqrt 2}$, 2nd order) — its stability lets us reach the same steady
state in **bigger** steps than implicit Euler:

In [ ]:
g = 1 - 2 ** -0.5                                            # SDIRK2 diagonal coefficient
dt_rk = 0.05                                                 # 5× the implicit-Euler step
dirk = DIRK(equation, [[g, 0], [1 - g, g]], [1 - g, g], dt_rk, fes)

gfd = GridFunction(fes); gfd.Set(CF(1), definedon=mesh.Boundaries("right"))
with TaskManager():
    for _ in range(int(tend / dt_rk + 0.5)):
        dirk.Step(gfd)
        
print(f"SDIRK2 (Δt={dt_rk}) marched to t={tend}:  u in [{min(gfd.vec):.2f}, {max(gfd.vec):.2f}]")
Draw(gfd, mesh, "SDIRK2 steady temperature", min=0, max=1, autoscale=False)

**Next:** the wind here was fixed and the equation **linear**. When the operator depends on the
solution itself, we step into **nonlinear** problems (unit 9).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("07-saddle-point", "7 · Mixed problems — the saddle point 🐎")
    _next = ("09-nonlinear-allencahn", "9 · Nonlinear problems — Allen–Cahn & Newton")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))